In [1]:
import os
import numpy as np
import librosa
import parselmouth
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Dataset folder
DATASET_PATH = "User_Voice"
users = sorted(os.listdir(DATASET_PATH))  # U001-U023

# Feature storage
X = []
y = []

def extract_behavioral_features(file_path):
    y_audio, sr = librosa.load(file_path, sr=16000, mono=True)
    y_audio, _ = librosa.effects.trim(y_audio, top_db=20)
    y_audio = y_audio / np.max(np.abs(y_audio)) if np.max(np.abs(y_audio)) > 0 else y_audio

    # Parselmouth features
    snd = parselmouth.Sound(file_path)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]
    pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    pitch_std = np.std(pitch_values) if len(pitch_values) > 0 else 0

    point_process = parselmouth.praat.call(snd, "To PointProcess (periodic, cc)", 75, 500)
    jitter_local = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    shimmer_local = parselmouth.praat.call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)

    energy = np.mean(y_audio**2)

    onset_env = librosa.onset.onset_strength(y=y_audio, sr=sr)
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr)
    duration_sec = librosa.get_duration(y=y_audio, sr=sr)
    speaking_rate = len(onsets)/duration_sec if duration_sec > 0 else 0

    mfccs = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
    mfcc_mean = np.mean(mfccs.T, axis=0)

    features = np.hstack([mfcc_mean, pitch_mean, pitch_std, jitter_local, shimmer_local, energy, speaking_rate])
    return features

# Iterate over users and files
for user_folder in users:
    user_path = os.path.join(DATASET_PATH, user_folder)
    files = os.listdir(user_path)
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(user_path, file)
            feats = extract_behavioral_features(file_path)
            X.append(feats)
            y.append(user_folder)

X = np.array(X)
y = np.array(y)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Encode labels
le_user = LabelEncoder()
y_encoded = le_user.fit_transform(y)

print("Feature extraction complete.")
print("X shape:", X_scaled.shape)
print("y shape:", y_encoded.shape)


Feature extraction complete.
X shape: (230, 26)
y shape: (230,)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------- Convert to Torch tensors ----------------
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# ---------------- Define model ----------------
class SpeakerNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(SpeakerNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.fc(x)

input_dim = X_scaled.shape[1]
num_classes = len(le_user.classes_)
model = SpeakerNet(input_dim, num_classes)

# ---------------- Training ----------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)
    acc = correct/total
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/total:.4f}, Accuracy: {acc:.4f}")

# Save model
torch.save(model.state_dict(), "speaker_model.pth")
print("Speaker model trained and saved.")


Epoch 1/50, Loss: 3.1028, Accuracy: 0.0565
Epoch 2/50, Loss: 2.9033, Accuracy: 0.3609
Epoch 3/50, Loss: 2.6532, Accuracy: 0.5652
Epoch 4/50, Loss: 2.3083, Accuracy: 0.5870
Epoch 5/50, Loss: 1.9096, Accuracy: 0.6435
Epoch 6/50, Loss: 1.5187, Accuracy: 0.7435
Epoch 7/50, Loss: 1.1847, Accuracy: 0.8043
Epoch 8/50, Loss: 0.9056, Accuracy: 0.9174
Epoch 9/50, Loss: 0.6921, Accuracy: 0.9609
Epoch 10/50, Loss: 0.5270, Accuracy: 0.9826
Epoch 11/50, Loss: 0.4047, Accuracy: 0.9826
Epoch 12/50, Loss: 0.3138, Accuracy: 0.9870
Epoch 13/50, Loss: 0.2456, Accuracy: 0.9913
Epoch 14/50, Loss: 0.1985, Accuracy: 0.9913
Epoch 15/50, Loss: 0.1628, Accuracy: 0.9957
Epoch 16/50, Loss: 0.1344, Accuracy: 0.9957
Epoch 17/50, Loss: 0.1110, Accuracy: 1.0000
Epoch 18/50, Loss: 0.0945, Accuracy: 1.0000
Epoch 19/50, Loss: 0.0802, Accuracy: 1.0000
Epoch 20/50, Loss: 0.0696, Accuracy: 1.0000
Epoch 21/50, Loss: 0.0601, Accuracy: 1.0000
Epoch 22/50, Loss: 0.0521, Accuracy: 1.0000
Epoch 23/50, Loss: 0.0464, Accuracy: 1.00

In [3]:
def verify_speaker_live(file_path, model, le_user, scaler, device='cpu'):
    feats = extract_behavioral_features(file_path)
    feats_scaled = scaler.transform(feats.reshape(1,-1))
    x_tensor = torch.tensor(feats_scaled, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        output = model(x_tensor)
        pred_idx = output.argmax(dim=1).cpu().item()
        predicted_user = le_user.inverse_transform([pred_idx])[0]
    return predicted_user

# Example usage
import sounddevice as sd
import soundfile as sf

def record_live(filename="live.wav", duration=10):
    print(f"Recording {duration} seconds...")
    audio = sd.rec(int(duration*16000), samplerate=16000, channels=1)
    sd.wait()
    sf.write(filename, audio, 16000)
    return filename

if __name__ == "__main__":
    live_file = record_live()
    speaker = verify_speaker_live(live_file, model, le_user, scaler, device)
    print("Identified Speaker:", speaker)


Recording 10 seconds...
Identified Speaker: U007
